# 02. Đánh giá bằng chứng và hallucination (offline, không cần GPU)

Đọc predictions của notebook 01, không gọi lại LLM. Mỗi câu trả lời có các *claim* (`{text, behavior, evidence ids}`), mỗi claim được gán nhãn
bởi verifier tất định (`pypi_rag/verification/verifier.py`) dựa trên **fact tĩnh của package** và **các item model đã thấy trong prompt**:

| Nhãn | Điều kiện |
|---|---|
| SUPPORTED | hành vi có trong fact tĩnh **và** một item của package được trích (C/G/F) cho thấy hành vi đó |
| UNSUPPORTED | không có fact, hoặc item được trích không cho thấy hành vi (trích sai), hoặc chỉ trích tri thức chung `K*` |
| CONTRADICTED | khẳng định X xảy ra khi phân tích đủ coverage và không có fact X nào; hoặc nói X không có trong khi AST cho thấy X |
| UNCHECKABLE | không xác định được hành vi nào trong claim |

Đây là kiểm tra **nhất quán với phân tích tĩnh**, không phải sự thật: claim không được hỗ trợ vẫn có thể đúng (phân tích tĩnh bỏ sót),
và claim được trích từ tài liệu sai vẫn có thể sai. Cell 06 xuất mẫu để người gán nhãn, đo độ tin cậy của verifier.

In [ ]:
# ==== 00a. Mount Drive + đường dẫn + helper ====
import os, sys, json, subprocess, time, shutil, glob, random
try:
    from google.colab import drive; drive.mount('/content/drive'); IN_COLAB = True
except ImportError:
    IN_COLAB = False                                        # chạy local để kiểm tra notebook (CPU)

DRIVE_BASE = '/content/drive/MyDrive/ThS/Research/colab' if IN_COLAB else os.environ.get('PYPI_RAG_BASE', os.path.abspath('../..'))  # SỬA NẾU KHÁC
CODE_DIR = f'{DRIVE_BASE}/pypi_malware_rag'                 # folder code này (upload nguyên folder lên Drive)
WORK     = os.environ.get('PYPI_RAG_WORK', f'{DRIVE_BASE}/pypi_rag_work')   # dữ liệu chuẩn hoá, KB, kết quả (giữ qua các phiên)
EXT      = '/content/external' if IN_COLAB else f'{WORK}/external'          # repo upstream + package nghi độc: đĩa máy ảo, KHÔNG để trên Drive
QUICK    = os.environ.get('PYPI_RAG_QUICK') == '1'          # 1 = kiểm tra nhanh: chỉ dữ liệu tổng hợp, reader dummy, không tải gì

os.environ['PYTHONDONTWRITEBYTECODE'] = '1'; os.environ['PYTHONIOENCODING'] = 'utf-8'; sys.dont_write_bytecode = True
for d in glob.glob(f'{CODE_DIR}/**/__pycache__', recursive=True): shutil.rmtree(d, ignore_errors=True)
for d in (WORK, f'{WORK}/data', f'{WORK}/kb', f'{WORK}/results', EXT): os.makedirs(d, exist_ok=True)
os.chdir(CODE_DIR); sys.path.insert(0, CODE_DIR)
assert os.path.exists('pypi_rag/cli.py'), 'chưa upload folder pypi_malware_rag lên Drive (hoặc sai DRIVE_BASE)'
for m in [m for m in sys.modules if m.split('.')[0] == 'pypi_rag']: del sys.modules[m]

CLI = f'{sys.executable} -m pypi_rag.cli'
def sh(cmd, check=True):
    """Chạy lệnh, in output trực tiếp; lỗi -> exception (check=True)."""
    print('$', cmd[:500]); t0 = time.time()
    p = subprocess.Popen(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         encoding='utf-8', errors='replace', cwd=CODE_DIR)
    for line in p.stdout: print(line, end='')
    rc = p.wait(); print(f'exit={rc}  ({(time.time() - t0) / 60:.1f} phút)')
    if check and rc: raise RuntimeError(f'lệnh lỗi (exit {rc}): {cmd[:200]}')
    return rc
def gate(cond, msg): print(('PASS  ' if cond else 'FAIL  ') + msg)
def load(out): return json.load(open(f'{out}/results.json', encoding='utf-8'))
import numpy as np, pandas as pd
pd.set_option('display.max_colwidth', 120); pd.set_option('display.width', 200)
print('code =', CODE_DIR, '| work =', WORK, '| colab =', IN_COLAB, '| quick =', QUICK)

In [ ]:
# ==== 00b. Nạp kết quả ====
from collections import Counter
from pypi_rag.schema import read_jsonl, load_samples
from pypi_rag.analysis import analyze_package
from pypi_rag.cli import _reverify, _load_preds
from pypi_rag.evaluation.classification import classification_metrics, risk_coverage
from pypi_rag.evaluation.evidence import claim_metrics
RUNS = sorted(d for d in glob.glob(f'{WORK}/results/r0[345]_test_*') if os.path.exists(f'{d}/results.json') and not d.endswith('_nodup'))
assert RUNS, 'chưa có kết quả r03/r04/r05 -> chạy notebook 01'
def data_of(run): return os.path.basename(load(run)['config']['data'].rstrip('/\\'))
def preds(run, m):
    p = f"{run}/predictions_{load(run)['config']['split']}_{m}.jsonl"
    return [json.loads(l) for l in open(p, encoding='utf-8')] if os.path.exists(p) else []
_SAMPLES = {}
def samples_of(run):
    """Mẫu (có mã nguồn) của đúng các sample id trong run; cache theo bộ dữ liệu."""
    data = data_of(run)
    if data not in _SAMPLES:
        _SAMPLES[data] = {s.sample_id: s for s in load_samples(f'{WORK}/data/{data}/samples.jsonl.gz')}
    ids = set(load(run)['sample_ids'])
    return {i: s for i, s in _SAMPLES[data].items() if i in ids}
print('\n'.join(os.path.basename(r) for r in RUNS))

## 01. Phân bố nhãn claim và độ chính xác trích dẫn theo method

In [ ]:
rows = []
for run in RUNS:
    R = load(run)
    for m in R['summaries']:
        cnt = Counter(c['label'] for p in preds(run, m) for c in p['verification'].get('claims', []))
        tot = sum(cnt.values()) or 1
        s = R['summaries'][m]
        rows.append(dict(run=os.path.basename(run).replace('r0', ''), method=m, claims=sum(cnt.values()),
                         **{k.lower(): cnt[k] / tot for k in ('SUPPORTED', 'UNSUPPORTED', 'CONTRADICTED', 'UNCHECKABLE')},
                         citation_acc=s['evidence/citation_accuracy'], invalid_cit=s['evidence/invalid_citation_rate'],
                         answers_w_unsupported=s['evidence/answers_with_unsupported']))
CLAIMS = pd.DataFrame(rows); display(CLAIMS.round(3))

## 02. Ablation xác minh và risk-coverage (plan mục 11)
E3 (không kiểm), `e4_cite` (abstain khi verdict độc không có trích dẫn hợp lệ), `e4_filter` (bỏ claim không có chứng cứ, giữ verdict),
`e4` (bỏ claim + abstain). Đọc ba cột cùng nhau: F1 *strict* (abstain tính là sai), F1 *answered* (chỉ câu đã trả lời) và *coverage*.

In [ ]:
import matplotlib.pyplot as plt
KEYS = ['coverage', 'strict/f1_malicious', 'answered/f1_malicious', 'conservative/f1_malicious', 'strict/fpr',
        'evidence/unsupported_claim_rate', 'evidence/contradicted_claim_rate', 'evidence/answers_with_unsupported', 'aurc']
for run in RUNS:
    R = load(run); S = R['summaries']
    ms = [m for m in ('e0', 'e2', 'e3', 'e4_cite', 'e4_filter', 'e4') if m in S]
    print('\n#####', os.path.basename(run)); display(pd.DataFrame({m: {k: S[m].get(k) for k in KEYS} for m in ms}).T.astype(float).round(3))
    lab = {i: s.label for i, s in samples_of(run).items()}
    fig, ax = plt.subplots(figsize=(6, 3.5))
    for m in [m for m in ('e0', 'e2', 'e3', 'e4') if m in S]:
        ps = [p for p in preds(run, m) if p['verdict'] != 'uncertain' and lab.get(p['sample_id']) in ('benign', 'malicious')]
        ps.sort(key=lambda p: -(p['confidence'] if p['confidence'] is not None else 0.5))
        err = np.cumsum([p['verdict'] != lab[p['sample_id']] for p in ps]) / np.arange(1, len(ps) + 1)
        ax.plot(np.arange(1, len(ps) + 1) / max(len(lab), 1), err, label=m)
    ax.set_xlabel('coverage (tỉ lệ package được trả lời, xếp theo confidence)'); ax.set_ylabel('tỉ lệ sai'); ax.legend(); ax.set_title(os.path.basename(run))
    plt.show()

## 03. Quét chính sách offline (chỉ để báo cáo)
Chính sách thật của E4 đã được chọn trên **val** (notebook 01, bước 02). Bảng này cho thấy độ nhạy của kết quả test theo ngưỡng, dựng lại từ output E3
mà không gọi LLM.

In [ ]:
SWEEP = {}
for run in RUNS:
    R = load(run)
    if 'e3' not in R['summaries']: continue
    S_by = samples_of(run); e3 = _load_preds(run, R['config']['split'], 'e3'); cache = {}
    rows = []
    for mc in (1, 2, 3):
        for sev in (None, 0.85, 0.9, 0.95):
            ps = _reverify(e3, S_by, 'abstain', mc, sev, cache, 'sweep')
            y = [S_by[p.sample_id].label for p in ps if S_by[p.sample_id].label in ('benign', 'malicious')]
            v = [p.verdict for p in ps if S_by[p.sample_id].label in ('benign', 'malicious')]
            cm = classification_metrics(y, v); ev = claim_metrics([p.verification for p in ps])
            rows.append(dict(min_critical=mc, conflict_severity=sev, coverage=cm['coverage'], strict_f1=cm['strict']['f1_malicious'],
                             answered_f1=cm['answered']['f1_malicious'], fpr=cm['strict']['fpr'], unsupported=ev['unsupported_claim_rate']))
    SWEEP[run] = pd.DataFrame(rows)
    print('\n#####', os.path.basename(run), '| chính sách dùng:', R['config'].get('policy')); display(SWEEP[run].round(3))

## 04. Phân tích lỗi theo nhóm (plan mục 13.6)
Nhóm (một package có thể thuộc nhiều nhóm): **obfuscation / không parse được**, **lành nhưng dùng API nhạy cảm**, **luồng dữ liệu nhiều hàm**
(có cạnh `interproc_summary`), **không có đường đồ thị** (E3 không tìm được path nào). Tỉ lệ sai = verdict khác nhãn (abstain tính là sai).
Mỗi run ghi `failure_cases.json` (một ca sai cho mỗi method) để dùng trong phần trình bày.

In [ ]:
def groups_for(s, f, tr):
    g = []
    b = f.behaviors()
    if f.parse_coverage() < 1 or 'obfuscation' in b: g.append('obfuscation/unparsable')
    if s.label == 'benign' and set(b) & {'network_send', 'process_exec', 'code_exec', 'env_access', 'file_write', 'install_hook'}:
        g.append('benign_sensitive_api')
    if any(x.extractor == 'interproc_summary' for x in f.flows): g.append('multi_function_flow')
    if tr is not None and tr.get('graph', {}).get('n_paths', 0) == 0: g.append('no_graph_path')
    return g or ['other']
ERR = {}
for run in RUNS:
    R = load(run); S_by = samples_of(run); sp = R['config']['split']
    facts = {i: analyze_package(s.public_view()) for i, s in S_by.items()}
    tr3 = {t['sample_id']: t for t in read_jsonl(f'{run}/traces_{sp}_e3.jsonl')} if os.path.exists(f'{run}/traces_{sp}_e3.jsonl') else {}
    grp = {i: groups_for(s, facts[i], tr3.get(i)) for i, s in S_by.items()}
    rows, fails = [], {}
    for m in R['summaries']:
        for p in preds(run, m):
            s = S_by.get(p['sample_id'])
            if s is None or s.label not in ('benign', 'malicious'): continue
            wrong = p['verdict'] != s.label
            for g in grp[p['sample_id']]: rows.append(dict(method=m, group=g, wrong=wrong))
            if wrong and m not in fails:
                fails[m] = dict(sample_id=p['sample_id'], label=s.label, verdict=p['verdict'], groups=grp[p['sample_id']],
                                explanation=p['explanation'][:500], claims=p['verification'].get('claims', [])[:5])
    df = pd.DataFrame(rows)
    ERR[run] = df.groupby(['group', 'method'])['wrong'].mean().unstack().round(3)
    print('\n#####', os.path.basename(run), '| tỉ lệ sai theo nhóm (số package mỗi nhóm:', dict(Counter(g for v in grp.values() for g in v)), ')')
    display(ERR[run])
    ERR[run].to_csv(f'{run}/error_groups.csv'); json.dump(fails, open(f'{run}/failure_cases.json', 'w'), indent=1)

## 05. Ví dụ cho phần trình bày
(a) KG-RAG giúp: package độc, E2 sai, E3 đúng nhờ một đường `FLOWS_TO`. (b) Thiếu cạnh làm hại: package độc, E3 sai, không có đường hoặc file không parse được.
(c) Cạnh gây hiểu lầm: package lành bị E3 gọi là độc. (d) E0 bịa: claim UNSUPPORTED / CONTRADICTED.

In [ ]:
def ev_of(p, prefix): return [e for e in p['evidence'] if e['evidence_id'].startswith(prefix)]
def show_case(title, run, sid, methods):
    S_by = samples_of(run); s = S_by[sid]
    print('=' * 110); print(title, '|', os.path.basename(run), '|', sid, '| nhãn:', s.label, '|', s.meta.get('difficulty', ''))
    for m in methods:
        p = next((x for x in preds(run, m) if x['sample_id'] == sid), None)
        if not p: continue
        print(f"--- {m}: verdict={p['verdict']} (raw {p['raw_verdict']}) | {p['explanation'][:300]}")
        for c in p['verification'].get('claims', [])[:4]: print(f"      claim [{c['label']}] {c['text'][:120]} | cites {c['cited']} | {c['why']}")
        if m.startswith('e3') or m == 'e4':
            for e in ev_of(p, 'G')[:2]: print('      ' + e['content'][:700].replace('\n', '\n      '))
CASES = {}
for run in RUNS:
    S_by = samples_of(run)
    P = {m: {p['sample_id']: p for p in preds(run, m)} for m in ('e0', 'e2', 'e3')}
    if not all(P.values()): continue
    lab = {i: s.label for i, s in S_by.items()}
    a = [i for i in P['e3'] if lab.get(i) == 'malicious' and P['e3'][i]['verdict'] == 'malicious' and P['e2'][i]['verdict'] != 'malicious'
         and any('FLOWS_TO path' in e['content'] for e in ev_of(P['e3'][i], 'G'))]
    b = [i for i in P['e3'] if lab.get(i) == 'malicious' and P['e3'][i]['verdict'] != 'malicious'
         and (any(e['meta'].get('empty') or e['meta'].get('coverage_note') for e in ev_of(P['e3'][i], 'G')))]
    c = [i for i in P['e3'] if lab.get(i) == 'benign' and P['e3'][i]['verdict'] == 'malicious']
    d = [i for i in P['e0'] if any(x['label'] in ('UNSUPPORTED', 'CONTRADICTED') for x in P['e0'][i]['verification'].get('claims', []))]
    CASES[run] = dict(kg_helps=a[:3], missing_edges=b[:3], misleading=c[:3], e0_hallucination=d[:3])
    for name, ids, ms in (('(a) KG-RAG giúp', a, ('e2', 'e3')), ('(b) thiếu cạnh', b, ('e2', 'e3')),
                          ('(c) cạnh gây hiểu lầm', c, ('e2', 'e3', 'e4')), ('(d) E0 bịa', d, ('e0',))):
        if ids: show_case(name, run, ids[0], ms)
        else: print('=' * 110 + f'\n{name} | {os.path.basename(run)}: không có ca nào')
json.dump({os.path.basename(k): v for k, v in CASES.items()}, open(f'{WORK}/results/cases.json', 'w'), indent=1)

## 06. Gán nhãn thủ công để kiểm verifier (plan mục 11 "annotate a stratified set")
Cell xuất `review/claims_review.csv`: mẫu phân tầng theo (run, method, nhãn tự động). Người gán điền cột `human_label`
(`SUPPORTED` / `UNSUPPORTED` / `CONTRADICTED`) và lưu thành `claims_review_A.csv` (và `_B.csv` nếu có người thứ hai).
Cell sau đo precision / recall / F1 của verifier khi phát hiện claim *không có chứng cứ* (UNSUPPORTED hoặc CONTRADICTED) và Cohen's kappa giữa hai người.
Chỉ khi verifier đạt mức chấp nhận được (ví dụ F1 >= 0.8) mới viết kết luận về hallucination dựa trên nhãn tự động.

In [ ]:
REVIEW = f'{WORK}/review'; os.makedirs(REVIEW, exist_ok=True)
CSV, PER_CELL = f'{REVIEW}/claims_review.csv', 8
if not os.path.exists(CSV):
    rng = random.Random(0); rows = []
    for run in RUNS:
        for m in ('e0', 'e2', 'e3', 'e4'):
            cl = [(p, c) for p in preds(run, m) for c in p['verification'].get('claims', [])]
            for lab in ('SUPPORTED', 'UNSUPPORTED', 'CONTRADICTED'):
                pool = [x for x in cl if x[1]['label'] == lab]
                for p, c in rng.sample(pool, min(PER_CELL, len(pool))):
                    ev = {e['evidence_id']: e['content'] for e in p['evidence']}
                    rows.append(dict(run=os.path.basename(run), method=m, sample_id=p['sample_id'], claim=c['text'],
                                     behaviors=','.join(c['behaviors']), negated=c['negated'], cited=','.join(c['cited']),
                                     evidence='\n---\n'.join(f'[{i}] ' + ev.get(i, '(không có trong prompt)')[:600] for i in c['cited']),
                                     auto_label=lab, human_label=''))
    pd.DataFrame(rows).to_csv(CSV, index=False); print('đã xuất', len(rows), 'claim ->', CSV)
def cohen_kappa(a, b):
    a, b = np.asarray(a), np.asarray(b); po = (a == b).mean()
    pe = sum((a == k).mean() * (b == k).mean() for k in set(a) | set(b)); return (po - pe) / max(1 - pe, 1e-9)
ann = {k: pd.read_csv(f'{REVIEW}/claims_review_{k}.csv') for k in ('A', 'B') if os.path.exists(f'{REVIEW}/claims_review_{k}.csv')}
for k, df in ann.items():
    df = df[df['human_label'].astype(str).str.upper().isin(['SUPPORTED', 'UNSUPPORTED', 'CONTRADICTED'])]
    y = df['human_label'].str.upper() != 'SUPPORTED'; p = df['auto_label'] != 'SUPPORTED'
    tp, fp, fn = int((y & p).sum()), int((~y & p).sum()), int((y & ~p).sum())
    pr, rc = tp / max(tp + fp, 1), tp / max(tp + fn, 1)
    print(f'người {k}: n={len(df)} | verifier phát hiện claim không có chứng cứ: P={pr:.3f} R={rc:.3f} F1={2 * pr * rc / max(pr + rc, 1e-9):.3f}')
if len(ann) == 2:
    m = ann['A'].merge(ann['B'], on=['run', 'method', 'sample_id', 'claim'], suffixes=('_a', '_b'))
    print("Cohen's kappa A-B:", round(cohen_kappa(m['human_label_a'].str.upper(), m['human_label_b'].str.upper()), 3))
if not ann: print('(chưa có file gán nhãn claims_review_A.csv)')

## 07. Kết luận nào được phép viết
Mỗi dòng [OK] phải trích kèm delta, CI 95%, p của bảng so sánh cặp (cùng sample id). [NO] là kết quả âm tính, vẫn phải báo cáo.

In [ ]:
lines = []
for run in RUNS:
    R = load(run); C = R['comparisons']; S = R['summaries']; name = os.path.basename(run)
    for k, txt in [('e1_vs_e0', 'RQ1 truy hồi giúp hơn LLM thuần'), ('e2_vs_e1', 'RQ2 hybrid hơn dense'),
                   ('e3_vs_e2', 'RQ3 KG-RAG hơn hybrid'), ('e3_vs_e3a', 'RQ3 cấu trúc đồ thị hơn cùng fact dạng câu'),
                   ('e4_vs_e3', 'RQ4 xác minh không làm giảm F1 (strict)')]:
        c = C.get(k)
        if not c: continue
        b = c['f1_malicious_strict']
        ok = b['ci_low'] > 0 if k != 'e4_vs_e3' else b['ci_low'] > -0.03
        lines.append(f"{'[OK]' if ok else '[NO]'} {name}: {txt}: dF1={b['delta']:+.3f} CI=[{b['ci_low']:+.3f}, {b['ci_high']:+.3f}] p={b['p_value']:.3f} n={c['n']}")
    if 'e3' in S and 'e4' in S:
        G = R.get('grounding_vs_e3') or {}
        if G.get('e4'):
            u3, u4 = G['e3']['unsupported_over_original_claims'], G['e4']['unsupported_over_original_claims']
            lines.append(f"{'[OK]' if u4 < u3 else '[NO]'} {name}: RQ4 claim không chứng cứ / claim gốc (cùng mẫu số) {u3:.3f} -> {u4:.3f} "
                         f"(claim giữ lại {G['e4']['claims_retained']:.2f}, coverage {G['e4']['coverage']:.2f})")
print('\n'.join(lines))
print('\nKhông kết luận "RAG / KG-RAG luôn tốt hơn" từ một bộ dữ liệu; nêu rõ loại dữ liệu (setup.py vs package đầy đủ) cho từng kết luận.')